In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [2]:
import os

BASE_DIR = os.path.abspath(
    os.path.join(os.getcwd(), "..")
)

RAW_DIR = os.path.join(BASE_DIR, "datasets", "raw", "ckd")
CLEANED_DIR = os.path.join(BASE_DIR, "datasets", "cleaned", "ckd")
PROCESSED_DIR = os.path.join(BASE_DIR, "datasets", "processed", "ckd")
MODEL_DIR = os.path.join(BASE_DIR, "models", "ckd")

os.makedirs(CLEANED_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Cleaned folder:", CLEANED_DIR)
print("Processed folder:", PROCESSED_DIR)
print("Model folder:", MODEL_DIR)

Raw folder: d:\Explainable-AI-CDSS\datasets\raw\ckd
Cleaned folder: d:\Explainable-AI-CDSS\datasets\cleaned\ckd
Processed folder: d:\Explainable-AI-CDSS\datasets\processed\ckd
Model folder: d:\Explainable-AI-CDSS\models\ckd


In [8]:
# Load CKD ARFF file
raw_file = os.path.join(
    RAW_DIR,
    "chronic_kidney_disease.arff"
)

with open(raw_file, "r") as file:
    lines = file.readlines()

# Find @data section
data_start = next(
    i for i, line in enumerate(lines)
    if line.strip().lower() == "@data"
)

data = []

for line in lines[data_start + 1:]:
    line = line.strip()

    if line and not line.startswith("%"):
        values = [x.strip() for x in line.split(",")]

        # CKD dataset should contain 25 columns
        if len(values) < 25:
            values = values + [np.nan] * (25 - len(values))
        elif len(values) > 25:
            values = values[:25]

        data.append(values)

columns = [
    "age", "bp", "sg", "al", "su",
    "rbc", "pc", "pcc", "ba",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wbcc", "rbcc",
    "htn", "dm", "cad", "appet", "pe", "ane",
    "class"
]

df = pd.DataFrame(data, columns=columns)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (400, 25)

Columns:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48,80,1.020,1,0,?,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,?,normal,notpresent,notpresent,?,...,38,6000,?,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,?,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [6]:
# Load CKD ARFF file
raw_file = os.path.join(
    RAW_DIR,
    "chronic_kidney_disease.arff"
)

with open(raw_file, "r") as file:
    lines = file.readlines()

# Find @data section
data_start = next(
    i for i, line in enumerate(lines)
    if line.strip().lower() == "@data"
)

# Read data rows
data = []

for line in lines[data_start + 1:]:
    line = line.strip()

    if line and not line.startswith("%"):
        values = [x.strip() for x in line.split(",")]

        # Keep only valid 25-column rows
        if len(values) == 25:
            data.append(values)

print("Number of valid rows:", len(data))
print("Values in first row:", len(data[0]))

# Correct CKD column names
columns = [
    "age", "bp", "sg", "al", "su",
    "rbc", "pc", "pcc", "ba",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wbcc", "rbcc",
    "htn", "dm", "cad", "appet", "pe", "ane",
    "class"
]

# Create DataFrame
df = pd.DataFrame(data, columns=columns)

print("\nDataset shape:", df.shape)

df.head()

Number of valid rows: 397
Values in first row: 25

Dataset shape: (397, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48,80,1.020,1,0,?,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,?,normal,notpresent,notpresent,?,...,38,6000,?,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,?,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [9]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDataset shape:")
print(df.shape)

Data types:
age      str
bp       str
sg       str
al       str
su       str
rbc      str
pc       str
pcc      str
ba       str
bgr      str
bu       str
sc       str
sod      str
pot      str
hemo     str
pcv      str
wbcc     str
rbcc     str
htn      str
dm       str
cad      str
appet    str
pe       str
ane      str
class    str
dtype: object

Missing values:
age      0
bp       0
sg       0
al       0
su       0
rbc      0
pc       0
pcc      0
ba       0
bgr      0
bu       0
sc       0
sod      0
pot      0
hemo     0
pcv      0
wbcc     0
rbcc     0
htn      0
dm       0
cad      0
appet    0
pe       0
ane      0
class    0
dtype: int64

Dataset shape:
(400, 25)


In [10]:
# Replace missing-value markers with NaN
df = df.replace(
    ["?", "\t?", " ?", "? "],
    np.nan
)

# Clean all string columns
for column in df.columns:
    if df[column].dtype == "object":
        df[column] = df[column].str.strip().str.lower()

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64


In [11]:
numeric_columns = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wbcc", "rbcc"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print(df[numeric_columns].dtypes)

age     float64
bp      float64
sg      float64
al      float64
su      float64
bgr     float64
bu      float64
sc      float64
sod     float64
pot     float64
hemo    float64
pcv     float64
wbcc    float64
rbcc    float64
dtype: object


In [12]:
categorical_columns = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad", "appet", "pe", "ane"
]

for column in categorical_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

print("Categorical values:")
for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].dropna().unique())

Categorical values:

rbc:
<StringArray>
['normal', 'abnormal']
Length: 2, dtype: string

pc:
<StringArray>
['normal', 'abnormal']
Length: 2, dtype: string

pcc:
<StringArray>
['notpresent', 'present']
Length: 2, dtype: string

ba:
<StringArray>
['notpresent', 'present']
Length: 2, dtype: string

htn:
<StringArray>
['yes', 'no']
Length: 2, dtype: string

dm:
<StringArray>
['yes', 'no', '']
Length: 3, dtype: string

cad:
<StringArray>
['no', 'yes']
Length: 2, dtype: string

appet:
<StringArray>
['good', 'poor', 'no']
Length: 3, dtype: string

pe:
<StringArray>
['no', 'yes', 'good']
Length: 3, dtype: string

ane:
<StringArray>
['no', 'yes']
Length: 2, dtype: string


In [13]:
df["class"] = df["class"].map({
    "ckd": 1,
    "notckd": 0
})

print("Target values:")
print(df["class"].value_counts())

print("\nTarget mapping:")
print("ckd = 1")
print("notckd = 0")

Target values:
class
1.0    250
0.0    149
Name: count, dtype: int64

Target mapping:
ckd = 1
notckd = 0


In [14]:
category_mappings = {
    "rbc": {"normal": 0, "abnormal": 1},
    "pc": {"normal": 0, "abnormal": 1},
    "pcc": {"notpresent": 0, "present": 1},
    "ba": {"notpresent": 0, "present": 1},
    "htn": {"no": 0, "yes": 1},
    "dm": {"no": 0, "yes": 1},
    "cad": {"no": 0, "yes": 1},
    "appet": {"good": 0, "poor": 1},
    "pe": {"no": 0, "yes": 1},
    "ane": {"no": 0, "yes": 1}
}

for column, mapping in category_mappings.items():
    df[column] = df[column].map(mapping)

print(df.dtypes)

age      float64
bp       float64
sg       float64
al       float64
su       float64
rbc      float64
pc       float64
pcc      float64
ba       float64
bgr      float64
bu       float64
sc       float64
sod      float64
pot      float64
hemo     float64
pcv      float64
wbcc     float64
rbcc     float64
htn      float64
dm       float64
cad      float64
appet    float64
pe       float64
ane      float64
class    float64
dtype: object


In [15]:
print("Dataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

Dataset shape:
(400, 25)

Missing values:
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         3
cad        2
appet      2
pe         2
ane        1
class      1
dtype: int64

Data types:
age      float64
bp       float64
sg       float64
al       float64
su       float64
rbc      float64
pc       float64
pcc      float64
ba       float64
bgr      float64
bu       float64
sc       float64
sod      float64
pot      float64
hemo     float64
pcv      float64
wbcc     float64
rbcc     float64
htn      float64
dm       float64
cad      float64
appet    float64
pe       float64
ane      float64
class    float64
dtype: object

First 5 rows:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,7.0,50.0,1.020,4.0,0.0,NaN,0.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,1.0,0.0,1.0,1.0
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,1.0,1.0,1.0,1.0
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [16]:
X = df.drop("class", axis=1)
y = df["class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features shape: (400, 24)
Target shape: (400,)

Target distribution:
class
1.0    250
0.0    149
Name: count, dtype: int64


In [17]:
# Remove rows where the target is missing
valid_target = y.notna()

X = X[valid_target].copy()
y = y[valid_target].copy()

# Convert target to integer
y = y.astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features shape: (399, 24)
Target shape: (399,)

Target distribution:
class
1    250
0    149
Name: count, dtype: int64


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

X_train shape: (319, 24)
X_test shape: (80, 24)

y_train distribution:
class
1    200
0    119
Name: count, dtype: int64

y_test distribution:
class
1    50
0    30
Name: count, dtype: int64


In [19]:
numeric_columns = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wbcc", "rbcc"
]

categorical_columns = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad", "appet", "pe", "ane"
]

print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))
print("Total features:", len(numeric_columns) + len(categorical_columns))

Numeric features: 14
Categorical features: 10
Total features: 24


In [20]:
numeric_imputer = SimpleImputer(strategy="median")

X_train_numeric = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_columns]),
    columns=numeric_columns,
    index=X_train.index
)

X_test_numeric = pd.DataFrame(
    numeric_imputer.transform(X_test[numeric_columns]),
    columns=numeric_columns,
    index=X_test.index
)

print("Numeric missing values after imputation:")
print(X_train_numeric.isnull().sum().sum())
print(X_test_numeric.isnull().sum().sum())

Numeric missing values after imputation:
0
0


In [21]:
categorical_imputer = SimpleImputer(strategy="most_frequent")

X_train_categorical = pd.DataFrame(
    categorical_imputer.fit_transform(
        X_train[categorical_columns]
    ),
    columns=categorical_columns,
    index=X_train.index
)

X_test_categorical = pd.DataFrame(
    categorical_imputer.transform(
        X_test[categorical_columns]
    ),
    columns=categorical_columns,
    index=X_test.index
)

print("Categorical missing values after imputation:")
print(X_train_categorical.isnull().sum().sum())
print(X_test_categorical.isnull().sum().sum())

Categorical missing values after imputation:
0
0


In [22]:
X_train_clean = pd.concat(
    [X_train_numeric, X_train_categorical],
    axis=1
)

X_test_clean = pd.concat(
    [X_test_numeric, X_test_categorical],
    axis=1
)

# Keep the same feature order
X_train_clean = X_train_clean[X.columns]
X_test_clean = X_test_clean[X.columns]

print("X_train_clean shape:", X_train_clean.shape)
print("X_test_clean shape:", X_test_clean.shape)

print("\nRemaining missing values:")
print("Train:", X_train_clean.isnull().sum().sum())
print("Test:", X_test_clean.isnull().sum().sum())

X_train_clean shape: (319, 24)
X_test_clean shape: (80, 24)

Remaining missing values:
Train: 0
Test: 0


In [23]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_clean),
    columns=X_train_clean.columns,
    index=X_train_clean.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_clean),
    columns=X_test_clean.columns,
    index=X_test_clean.index
)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("\nFirst 5 rows of scaled training data:")
display(X_train_scaled.head())

X_train_scaled shape: (319, 24)
X_test_scaled shape: (80, 24)

First 5 rows of scaled training data:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
67,-0.351555,0.220380,0.419716,1.635620,-0.348282,-0.356663,2.015811,-0.37321,-0.25863,3.949121,...,0.022796,0.135499,-0.102788,0.066884,-0.761057,-0.685599,-0.291606,2.035967,-0.471405,-0.426006
112,0.180503,-1.214339,-0.510072,1.635620,-0.348282,2.803767,2.015811,-0.37321,-0.25863,-0.297937,...,-0.625204,-0.724071,-0.102788,0.066884,-0.761057,-0.685599,-0.291606,-0.491167,-0.471405,-0.426006
6,1.008150,-0.496979,-1.439860,-0.679895,-0.348282,-0.356663,-0.496078,-0.37321,-0.25863,-0.591319,...,-0.049204,-0.355684,-0.102788,0.066884,-0.761057,-0.685599,-0.291606,-0.491167,-0.471405,-0.426006
148,1.067267,-1.214339,0.419716,-0.679895,-0.348282,-0.356663,-0.496078,-0.37321,-0.25863,0.400592,...,0.022796,0.135499,-0.102788,0.066884,1.313962,-0.685599,-0.291606,2.035967,-0.471405,-0.426006
188,-2.538906,0.220380,0.419716,-0.679895,-0.348282,-0.356663,-0.496078,-0.37321,-0.25863,-0.870731,...,-0.121204,-0.110093,-0.102788,0.066884,-0.761057,-0.685599,-0.291606,-0.491167,-0.471405,-0.426006


In [24]:
X_train_scaled.to_csv(
    os.path.join(PROCESSED_DIR, "x_train.csv"),
    index=False
)

X_test_scaled.to_csv(
    os.path.join(PROCESSED_DIR, "x_test.csv"),
    index=False
)

y_train.to_csv(
    os.path.join(PROCESSED_DIR, "y_train.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(PROCESSED_DIR, "y_test.csv"),
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [25]:
import joblib

scaler_path = os.path.join(
    MODEL_DIR,
    "scaler.pkl"
)

joblib.dump(scaler, scaler_path)

print("CKD scaler saved successfully!")
print("Location:", scaler_path)

CKD scaler saved successfully!
Location: d:\Explainable-AI-CDSS\models\ckd\scaler.pkl
